# Diagnostic FP8 — Qwen3.6-27B-FP8

**Objectif :** isoler la cause racine de la génération incohérente (chinois / charabia).

## Hypothèse confirmée par le diagnostic
Le kernel FP8 est chargé mais **jamais appelé** pendant l'inférence (compteurs = 0/0/0).  
PyTorch fait du matmul FP8 natif sur H100 **sans appliquer les weight_scale_inv** → valeurs fausses → tokens incohérents.

## Plan
| Étape | Test | Attendu |
|-------|------|---------|
| A | Chargement BF16 (sans FP8) | Sortie cohérente → pipeline sain |
| B | Patch FP8Linear post-chargement | Compteurs kernel > 0, sortie cohérente |

**Exécuter les cellules dans l'ordre. Ne pas sauter d'étape.**

## 0. Environnement

In [ ]:
import os, sys, time, importlib, importlib.util, gc
import torch
import transformers
from pathlib import Path

print('Python      :', sys.version.split()[0])
print('PyTorch     :', torch.__version__)
print('CUDA        :', torch.version.cuda)
print('Transformers:', transformers.__version__)
print('GPU         :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun')
print('VRAM totale :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

MODEL_PATH       = Path('/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main')
LOCAL_FP8_KERNEL = Path('/mnt/finegrained-fp8/build/torch-cuda')

assert MODEL_PATH.exists(),       f'Modèle introuvable : {MODEL_PATH}'
assert LOCAL_FP8_KERNEL.exists(), f'Kernel introuvable : {LOCAL_FP8_KERNEL}'
assert torch.cuda.is_available(), 'GPU CUDA requis'

print('\n✅ Environnement OK')

---
## ÉTAPE A — Test BF16 (sans FP8)

Chargement du modèle en **bfloat16 pur**, en ignorant la config de quantification FP8.  
Si la sortie est cohérente → le problème est bien dans le chemin FP8, pas dans le modèle ou le pipeline.

In [ ]:
os.environ['HF_HUB_OFFLINE']    = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR'] = '1'

from transformers import AutoProcessor, AutoModelForMultimodalLM

print('Chargement processor...')
processor_a = AutoProcessor.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
)

print('Chargement modèle en BF16 (quantization_config ignorée)...')
t0 = time.time()
model_a = AutoModelForMultimodalLM.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    quantization_config=None,   # ignore la config FP8
)
model_a.eval()

print(f'✅ Modèle BF16 chargé en {time.time()-t0:.1f}s')
print('Dtype premier paramètre :', next(model_a.parameters()).dtype)
print('VRAM allouée :', round(torch.cuda.memory_allocated()/1024**3, 2), 'GB')

In [ ]:
# ── Test texte minimal ────────────────────────────────────────────────────────
PROMPT_TEST = 'Réponds uniquement par : TEST OK'

messages = [
    {'role': 'system',  'content': 'Tu es un assistant. Réponds uniquement en français.'},
    {'role': 'user',    'content': PROMPT_TEST},
]

try:
    text_a = processor_a.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
except TypeError:
    text_a = processor_a.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

inputs_a = processor_a(text=[text_a], return_tensors='pt').to(model_a.device)

print(f'Tokens en entrée : {inputs_a["input_ids"].shape[-1]}')
print('Génération...')

t0 = time.time()
with torch.inference_mode():
    out_a = model_a.generate(**inputs_a, max_new_tokens=30, do_sample=False)

generated_a = out_a[0][inputs_a['input_ids'].shape[-1]:]
response_a  = processor_a.decode(generated_a, skip_special_tokens=True)

print(f'\nTemps : {time.time()-t0:.1f}s')
print('=' * 60)
print('RÉPONSE BF16 :', response_a)
print('=' * 60)

if 'TEST OK' in response_a.upper() or any(c.isalpha() and ord(c) < 128 for c in response_a):
    print('\n✅ ÉTAPE A RÉUSSIE — sortie cohérente en BF16')
    print('   → Le problème est bien isolé au chemin FP8')
    print('   → Passer à l\'ÉTAPE B')
else:
    print('\n⚠️  ÉTAPE A — sortie incohérente même en BF16')
    print('   → Problème plus profond (architecture, tokenizer, ou version transformers)')
    print('   → NE PAS passer à l\'ÉTAPE B avant investigation')

In [ ]:
# Libérer la VRAM avant l'étape B
del model_a
gc.collect()
torch.cuda.empty_cache()
print('VRAM libérée :', round(torch.cuda.memory_allocated()/1024**3, 2), 'GB')

---
## ÉTAPE B — Patch FP8Linear post-chargement

**Exécuter uniquement si l'ÉTAPE A a donné une sortie cohérente.**

Le patch actuel cible `tf_fp8.lazy_load_kernel` (niveau module), mais les instances `FP8Linear`  
déjà créées ont capturé l'ancienne référence au moment de leur `__init__`.  
On patche ici **directement chaque instance** après chargement.

In [ ]:
# ── 1. Charger le kernel FP8 local ────────────────────────────────────────────
PKG_NAME = '_finegrained_fp8_local'

if PKG_NAME in sys.modules:
    local_fp8_kernel = sys.modules[PKG_NAME]
    print('Kernel déjà en cache.')
else:
    spec = importlib.util.spec_from_file_location(
        PKG_NAME,
        str(LOCAL_FP8_KERNEL / '__init__.py'),
        submodule_search_locations=[str(LOCAL_FP8_KERNEL)],
    )
    local_fp8_kernel = importlib.util.module_from_spec(spec)
    local_fp8_kernel.__package__ = PKG_NAME
    sys.modules[PKG_NAME] = local_fp8_kernel
    spec.loader.exec_module(local_fp8_kernel)

for fn in ('matmul_2d', 'matmul_batched', 'matmul_grouped'):
    assert hasattr(local_fp8_kernel, fn), f'Fonction manquante : {fn}'

print('✅ Kernel chargé :', LOCAL_FP8_KERNEL)
print('   matmul_2d    :', local_fp8_kernel.matmul_2d)
print('   matmul_batched:', local_fp8_kernel.matmul_batched)
print('   matmul_grouped:', local_fp8_kernel.matmul_grouped)

In [ ]:
# ── 2. Installer des compteurs sur les fonctions du kernel ────────────────────
_call_counts = {'matmul_2d': 0, 'matmul_batched': 0, 'matmul_grouped': 0}
_original    = {
    'matmul_2d':     local_fp8_kernel.matmul_2d,
    'matmul_batched': local_fp8_kernel.matmul_batched,
    'matmul_grouped': local_fp8_kernel.matmul_grouped,
}

def _make_counter(name, fn):
    def wrapper(*args, **kwargs):
        _call_counts[name] += 1
        return fn(*args, **kwargs)
    wrapper.__name__ = name
    return wrapper

local_fp8_kernel.matmul_2d      = _make_counter('matmul_2d',     _original['matmul_2d'])
local_fp8_kernel.matmul_batched = _make_counter('matmul_batched', _original['matmul_batched'])
local_fp8_kernel.matmul_grouped = _make_counter('matmul_grouped', _original['matmul_grouped'])

print('✅ Compteurs installés')
print('Compteurs actuels :', _call_counts)

In [ ]:
# ── 3. Patch niveau module Transformers ───────────────────────────────────────
import transformers.integrations.finegrained_fp8 as tf_fp8

_loader = lambda *a, **kw: local_fp8_kernel
tf_fp8.lazy_load_kernel                = _loader
tf_fp8._load_finegrained_fp8_kernel    = _loader
tf_fp8.load_finegrained_fp8_kernel     = _loader

for _attr in ('_load_finegrained_fp8_kernel', 'load_finegrained_fp8_kernel'):
    _fn = getattr(tf_fp8, _attr, None)
    if _fn and hasattr(_fn, 'cache_clear'):
        _fn.cache_clear()

print('✅ Patch module tf_fp8 OK')

In [ ]:
# ── 4. Charger le modèle FP8 ──────────────────────────────────────────────────
print('Chargement Qwen3.6-27B-FP8...')
t0 = time.time()

processor_b = AutoProcessor.from_pretrained(
    str(MODEL_PATH), local_files_only=True, trust_remote_code=True
)
model_b = AutoModelForMultimodalLM.from_pretrained(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=True,
    device_map='auto',
    dtype='auto',
)
model_b.eval()

print(f'✅ Modèle FP8 chargé en {time.time()-t0:.1f}s')
print('VRAM allouée :', round(torch.cuda.memory_allocated()/1024**3, 2), 'GB')

In [ ]:
# ── 5. Patch direct sur chaque instance FP8Linear ────────────────────────────
# C'est le patch clé : les instances ont capturé l'ancienne référence
# au moment du from_pretrained. On les corrige après coup.

patched_count = 0
fp8_attrs     = ('_kernel', 'kernel', '_fp8_kernel', 'fp8_kernel')
fp8_loaders   = ('load_kernel', '_load_kernel', 'get_kernel', '_get_kernel')

for name, module in model_b.named_modules():
    cls_name = type(module).__name__
    is_fp8   = 'FP8' in cls_name or 'fp8' in cls_name.lower() or 'Fp8' in cls_name

    if not is_fp8:
        continue

    patched_here = False
    for attr in fp8_attrs:
        if hasattr(module, attr):
            setattr(module, attr, local_fp8_kernel)
            patched_here = True
    for attr in fp8_loaders:
        if hasattr(module, attr):
            setattr(module, attr, _loader)
            patched_here = True

    if patched_here:
        patched_count += 1

print(f'Modules FP8 patchés : {patched_count}')

# Inventaire des types FP8 trouvés dans le modèle
fp8_types = set()
for _, m in model_b.named_modules():
    cls = type(m).__name__
    if 'FP8' in cls or 'fp8' in cls.lower() or 'Fp8' in cls:
        fp8_types.add(cls)

print('Types FP8 trouvés :', fp8_types if fp8_types else '(aucun — vérifier architecture)')

In [ ]:
# ── 6. Test texte minimal avec compteurs ─────────────────────────────────────
_call_counts['matmul_2d']      = 0
_call_counts['matmul_batched'] = 0
_call_counts['matmul_grouped'] = 0

messages = [
    {'role': 'system', 'content': 'Tu es un assistant. Réponds uniquement en français.'},
    {'role': 'user',   'content': 'Réponds uniquement par : TEST OK'},
]

try:
    text_b = processor_b.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
except TypeError:
    text_b = processor_b.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

inputs_b = processor_b(text=[text_b], return_tensors='pt').to(model_b.device)
print(f'Tokens en entrée : {inputs_b["input_ids"].shape[-1]}')
print('Compteurs AVANT génération :', dict(_call_counts))
print('Génération...')

t0 = time.time()
with torch.inference_mode():
    out_b = model_b.generate(**inputs_b, max_new_tokens=30, do_sample=False)

generated_b = out_b[0][inputs_b['input_ids'].shape[-1]:]
response_b  = processor_b.decode(generated_b, skip_special_tokens=True)

print(f'Temps : {time.time()-t0:.1f}s')
print('Compteurs APRÈS génération :', dict(_call_counts))
print('=' * 60)
print('RÉPONSE FP8 :', response_b)
print('=' * 60)

In [ ]:
# ── 7. Verdict ────────────────────────────────────────────────────────────────
kernel_called = sum(_call_counts.values()) > 0
coherent      = any(c.isalpha() and ord(c) < 256 for c in response_b)

print('\n══ VERDICT ══')
print(f'Kernel appelé       : {"✅ OUI" if kernel_called else "❌ NON (0 appels)"}')
print(f'Sortie cohérente    : {"✅ OUI" if coherent else "❌ NON (charabia/chinois)"}')
print()

if kernel_called and coherent:
    print('✅ FP8 fonctionne correctement avec le patch post-chargement.')
    print('   → Intégrer ce patch dans pipeline_ocr_v13_qwen36_27b_fp8.ipynb')
elif not kernel_called and coherent:
    print('⚠️  Kernel non appelé MAIS sortie cohérente.')
    print('   → Transformers dequantise en BF16 en fallback (plus lent mais fonctionnel).')
    print('   → Le pipeline peut fonctionner. Vérifier les performances.')
elif kernel_called and not coherent:
    print('❌ Kernel appelé mais sortie incohérente.')
    print('   → Le kernel produit des valeurs fausses.')
    print('   → Vérifier la version du kernel vs la version des poids FP8.')
else:
    print('❌ Kernel non appelé ET sortie incohérente.')
    print('   → Le patch post-chargement n\'a pas suffi.')
    print('   → Investiguer le forward() de FP8Linear dans transformers', transformers.__version__)

---
## ÉTAPE C — Inspection du forward FP8Linear

**Exécuter uniquement si kernel = 0 appels après l'étape B.**

Inspecte le code source du `forward()` pour comprendre comment le kernel est appelé.

In [ ]:
import inspect

# Trouver un module FP8Linear dans le modèle
fp8_module = None
for name, module in model_b.named_modules():
    cls = type(module).__name__
    if 'FP8' in cls or 'fp8' in cls.lower():
        fp8_module = module
        print(f'Module trouvé : {name} ({cls})')
        break

if fp8_module is None:
    print('Aucun module FP8 trouvé — le modèle utilise peut-être compressed-tensors directement.')
    print('Vérifier compressed_tensors dans le modèle...')
    for name, module in model_b.named_modules():
        cls = type(module).__name__
        if 'compressed' in cls.lower() or 'quant' in cls.lower() or 'linear' in cls.lower():
            print(f'  {cls} → {name}')
            break
else:
    print('\nAttributs du module FP8 :')
    for attr in dir(fp8_module):
        if not attr.startswith('__'):
            val = getattr(fp8_module, attr, None)
            if not callable(val) or 'kernel' in attr.lower() or 'fp8' in attr.lower():
                print(f'  {attr}: {type(val).__name__} = {str(val)[:80]}')

    print('\nSource forward() :')
    try:
        print(inspect.getsource(fp8_module.forward))
    except Exception as e:
        print('(impossible de récupérer la source :', e, ')')

---
## Résumé des résultats

Remplir après exécution :

| Test | Résultat | Conclusion |
|------|----------|------------|
| A — BF16 | | |
| B — FP8 kernel patché | Kernel appelés : ? | |
| B — FP8 sortie | | |